# Revenue per customer is higher than the ledger

**Track:** debug-the-ai | **Difficulty:** easy | **Tags:** joins, row duplication, aggregation

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("001-orders-payments-join")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "orders": """
UEFSMRUEFeASFdIJTBWsAhUAEgAAsAkIiRMABQEAig0IAIsNCACMDQgAjQ0IAI4NCACPDQgAkA0IAJENCACSDQgAkw0IAJQNCACV
DQgAlg0IAJcNCACYDQgAmQ0IAJoNCACbDQgAnA0IAJ0NCACeDQgAnw0IAKANCAChDQgAog0IAKMNCACkDQgApQ0IAKYNCACnDQgA
qA0IAKkNCACqDQgAqw0IAKwNCACtDQgArg0IAK8NCACwDQgAsQ0IALINCACzDQgAtA0IALUNCAC2DQgAtw0IALgNCAC5DQgAug0I
ALsNCAC8DQgAvQ0IAL4NCAC/DQgAwA0IAMENCADCDQgAww0IAMQNCADFDQgAxg0IAMcNCADIDQgAyQ0IAMoNCADLDQgAzA0IAM0N
CADODQgAzw0IANANCADRDQgA0g0IANMNCADUDQgA1Q0IANYNCADXDQgA2A0IANkNCADaDQgA2w0IANwNCADdDQgI3hMABQEA3w0I
AOANCADhDQgA4g0IAOMNCADkDQgA5Q0IAOYNCADnDQgA6A0IAOkNCADqDQgA6w0IAOwNCADtDQgA7g0IAO8NCADwDQgA8Q0IAPIN
CADzDQgA9A0IAPUNCAD2DQgA9w0IAPgNCAD5DQgA+g0IAPsNCAD8DQgA/Q0IAP4NCAD/DQgEABQlDwQAAQ0IAAINCAADDQgABA0I
AAUNCAAGDQgABw0IAAgNCAAJDQgACg0IAAsNCAAMDQgADQ0IAA4NCAAPDQgAEA0IABENCAASDQgAEw0IABQNCAAVDQgAFg0IABcN
CAAYDQgAGQ0IABoNCAAbDQgAHA0IPB0UAAAAAAAAHhQAAAAAAAAVABXCAhXKAiwVrAIVEBUGFQYcGAgeFAAAAAAAABgIiRMAAAAA
AAAWACgIHhQAAAAAAAAYCIkTAAAAAAAAEREAAAChAfCgAwAAAKwCAQgnAAECAwQFBgcICQoLDA0ODxAREhMUFRYXGBkaGxwdHh8g
ISIjJCUmJygpKissLS4vMDEyMzQ1Njc4OTo7PD0+P0BBQkNERUZHSElKS0xNTk9QUVJTVFVWV1hZWltcXV5fYGFiY2RlZmdoaWpr
bG1ub3BxcnN0dXZ3eHl6e3x9fn+AgYKDhIWGh4iJiouMjY6PkJGSk5SVAAAVBBXwBBXaAkwVThUAEgAAuAIEeAAJAQCLCQcEAG8N
CABlDQgAfA0IAIUNCAB6DQgAiQ0IAGwNCABnDQgAdQ0IBHQACQEAcQkHBAB9DQgAew0IAHINCABmDQgAag0IAIgNCABrDQgAgQ0I
AHMNCAB5DQgAdw0IAG0NCAB2DQgEaQAJAQCACQcEAG4NCACHDQgAig0IAH4NCACCDQgAaA0IAIwNCACGDQgAhA0IPH8AAAAAAAAA
cAAAAAAAAAAVABX2ARX8ASwVrAIVEBUGFQYcGAiMAAAAAAAAABgIZQAAAAAAAAAWACgIjAAAAAAAAAAYCGUAAAAAAAAAEREAAAB7
8HoDAAAArAIBBidAIAxEYRxIoiwI0zjMo0DEUiiRNA0KVRlSY00XVkmZpgjDNmHVlCwPF2Ed0XhC8ikNmGQRV2zh1IhXc3mQpwlj
doCFIBBIlDkEGAhbV5GesoVGl2Vd1HFV9WiR9GmI8zEVmYFS8UBd2XEPaIFblVmICQAVBBXoBBXyBEwVmgEVABIAALQC9DMBz04A
AMNOAADNTgAAk04AAM5OAACzTgAAiU4AAL5OAACbTgAAhE4AAKNOAADATgAAqE4AAKdOAACmTgAAjE4AAI9OAAB9TgAAoE4AAJxO
AAChTgAAmU4AAKxOAADHTgAAnU4AAI5OAACXTgAAfE4AANJOAACGTgAAf04AAIpOAACxTgAAjU4AALtOAACBTgAAgE4AAMJOAACq
TgAAt04AALROAACDTgAApE4AAIJOAADBTgAAek4AAMROAACyTgAAok4AAJpOAADLTgAAnk4AAK9OAACITgAAkU4AAL1OAAC1TgAA
0U4AAMlOAACwTgAAuk4AAMhOAAC4TgAAmE4AAHlOAACWTgAAuU4AALxOAAB+TgAAi04AAMpOAACuTgAAn04AAMxOAAC2TgAAxk4A
AKlOAAAVABWcAhWkAiwVrAIVEBUGFQYcGATSTgAAGAR5TgAAFgAoBNJOAAAYBHlOAAAREQAAAI4B8I0DAAAArAIBByeAgGBAKBgO
AkRCsWA0HA8IISJRTCCUSkNhuWAyms2F0/F8QCGR5jMilUwnVMqjOmwMmRWr9XC9YLGUbEaT1Fy2GwenGj0Wl4Emh9HJMpwdIIBT
FHg9jCTh+42Aj0LQJByqDgli8RLEGE2lVhAQOA5nCcbGiDQlQYeawTVpGXgC5ZLJUQEAFQQV0gsV3AtMFaoCFQASAADpBfToAgAA
DBCrAAAeB4AAAB18ogAAIcoyAAAglAoAAB9chAAAEcDsAAAjewIAAAV7hAAADmP8AAAgBN8AAAbYJgAAAMksAAAdbZgAAAJMegAA
HEyHAAAMI9AAABTrFAAAJQ5UAAAX1lgAAAuuoAAADjvAAAAR3KcAAB1VLgAAFh9IAAAChhcAABXeDAAAH3YaAAAjKiYAABzCTgAA
CtBbAAAhPvAAACHxpgAAAp0kAAAD8icAABOA9wAAFY+IAAAcLIAAABeJMgAAHu/rAAAlfN8AABiIeAAADeOqAAAH2DQAABUMRwAA
CqVjAAATUt8AABVBBAAAAcGbAAAGr1QAACBxEwAADU/PAAATQqoAAALoVgAADKqUAAAX+xAAAB6Y0AAAHehAAAACW1IAABVOrwAA
FhP0AAANuRYAAAx/NwAAAMLsAAARUi4AAAxfLwAAI+VzAAALYnMAACWZlgAADN9QAAAlclQAABqJKgAADx2IAAAGodoAAAaeIwAA
GD13AAAJhYAAABRXBwAAHHI6AAAVHLAAACVOyAAAH5P8AAAI/lgAAB3+5wAAI3QrAAAdyvQAAAfA9gAAD68KAAAUbeIAACUH4gAA
DcdYAAAWGWsAABGzowAAFOLfAAAEtOIAAABckwAAJiCLAAAWNe8AABvYIAAAC4mEAAAkIjIAABckOAAADh4PAAAOcj4AABahvwAA
FNeMAAAhfKgAAB3QOgAAFyYsAAAfd3cAAAIZHAAAFgBsAAAQhbAAAAyDgwAAE244AAAXXEYAACIuyAAAFYTKAAAYdoAAAA8ZCgAA
HJl8AAAeqD4AACYU1AAAEouoAAAJFcgAABlklgAAJaHKAAAUCCAAACHyCgAAH0gCAAAXhIIAABjvNAAAFvwuAAAFQU8AABaJiAAA
IItyAAAXCgwAAByITAAACxnMAAAHibAAABI25AAAJWPfAAAd/JAAABpgJgAAFGK/AAAHuCwAAB4sagAABgYwAAAW+s8VABXCAhXK
AiwVrAIVEBUGFQYcGAUAACYgixgFAAAAXJMWACgFAAAmIIsYBQAAAFyTEREAAAChAfCgAwAAAKwCAQgnAAECAwQFBgcICQoLDA0O
DxAREhMUFRYXGBkaGxwdHh8gISIjJCUmJygpKissLS4vMDEyMzQ1Njc4OTo7PD0+P0BBQkNERUZHSElKS0xNTk9QUVJTVFVWV1hZ
WltcXV5fYGFiY2RlZmdoP2lqa2xtbm9wcXJzdHV2d3h5ent8fX5/gIGCg4SFhoeIiYqLjI2Oj5CRkpOUAAAVBBlcNQAYBnNjaGVt
YRUIABUEJQIYCG9yZGVyX2lkABUEJQIYC2N1c3RvbWVyX2lkABUCJQIYCm9yZGVyX2RhdGUlDExsAAAAFQ4VChUCGAxvcmRlcl9h
bW91bnQlChUEFRQsXBUEFRQAAAAWrAIZHBlMJgAcFQQZNQAGEBkYCG9yZGVyX2lkFQIWrAIWyBYWwg0m/AkmCBwYCB4UAAAAAAAA
GAiJEwAAAAAAABYAKAgeFAAAAAAAABgIiRMAAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYArAIAAAAmABwVBBk1AAYQGRgL
Y3VzdG9tZXJfaWQVAhasAhaKCBb6BSbEECbKDRwYCIwAAAAAAAAAGAhlAAAAAAAAABYAKAiMAAAAAAAAABgIZQAAAAAAAAAREQAZ
LBUEFQAVAgAVABUQFQIAPCkGGSYArAIAAAAmABwVAhk1AAYQGRgKb3JkZXJfZGF0ZRUCFqwCFooIFpwIJtgYJsQTHBgE0k4AABgE
eU4AABYAKATSTgAAGAR5TgAAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAKwCAAAAJgAcFQ4ZNQAGEBkYDG9yZGVyX2Ftb3VudBUC
FqwCFqIPFrQPJt4nJuAbHBgFAAAmIIsYBQAAAFyTFgAoBQAAJiCLGAUAAABckxERABksFQQVABUCABUAFRAVAgA8KQYZJgCsAgAA
ABb+NRasAiYIFowrABkcGAxBUlJPVzpzY2hlbWEYrAMvLy8vL3pnQkFBQVFBQUFBQUFBS0FBd0FCZ0FGQUFnQUNnQUFBQUFCQkFB
TUFBQUFDQUFJQUFBQUJBQUlBQUFBQkFBQUFBUUFBQURNQUFBQWhBQUFBRWdBQUFBRUFBQUFWUC8vL3dBQUFRY1FBQUFBS0FBQUFB
UUFBQUFBQUFBQURBQUFBRzl5WkdWeVgyRnRiM1Z1ZEFBQUFBQUlBQXdBQkFBSUFBZ0FBQUFLQUFBQUFnQUFBSlQvLy84QUFBRUlF
QUFBQUNRQUFBQUVBQUFBQUFBQUFBb0FBQUJ2Y21SbGNsOWtZWFJsQUFBQUFBWUFDQUFHQUFZQUFBQUFBQUFBelAvLy93QUFBUUlR
QUFBQUhBQUFBQVFBQUFBQUFBQUFDd0FBQUdOMWMzUnZiV1Z5WDJsa0FMei8vLzhBQUFBQlFBQUFBQkFBRkFBSUFBWUFCd0FNQUFB
QUVBQVFBQUFBQUFBQkFoQUFBQUFrQUFBQUJBQUFBQUFBQUFBSUFBQUFiM0prWlhKZmFXUUFBQUFBQ0FBTUFBZ0FCd0FJQUFBQUFB
QUFBVUFBQUFBPQAYIHBhcnF1ZXQtY3BwLWFycm93IHZlcnNpb24gMjUuMC4xGUwcAAAcAAAcAAAcAAAAGAQAAFBBUjE=
""",
    "payments": """
UEFSMRUEFdAWFcYLTBXqAhUAEgAAqAsMgTgBAAEBAIINCACDDQgAhA0IAIUNCACGDQgAhw0IAIgNCACJDQgAig0IAIsNCACMDQgA
jQ0IAI4NCACPDQgAkA0IAJENCACSDQgAkw0IAJQNCACVDQgAlg0IAJcNCACYDQgAmQ0IAJoNCACbDQgAnA0IAJ0NCACeDQgAnw0I
AKANCAChDQgAog0IAKMNCACkDQgApQ0IAKYNCACnDQgAqA0IAKkNCACqDQgAqw0IAKwNCACtDQgArg0IAK8NCACwDQgAsQ0IALIN
CACzDQgAtA0IALUNCAC2DQgAtw0IALgNCAC5DQgAug0IALsNCAC8DQgAvQ0IAL4NCAC/DQgAwA0IAMENCADCDQgAww0IAMQNCADF
DQgAxg0IAMcNCADIDQgAyQ0IAMoNCADLDQgAzA0IAM0NCADODQgAzw0IANANCADRDQgA0g0IANMNCADUDQgA1Q0IANYNCADXDQgA
2A0IANkNCADaDQgA2w0IANwNCADdDQgA3g0IAN8NCADgDQgA4Q0IAOINCADjDQgA5A0IAOUNCADmDQgA5w0IAOgNCADpDQgA6g0I
AOsNCADsDQgA7Q0IAO4NCADvDQgA8A0IAPENCADyDQgA8w0IAPQNCAD1DQgA9g0IAPcNCAD4DQgA+Q0IAPoNCAD7DQgA/A0IAP0N
CAD+DQgA/w0IBAA5afgAAQ0IAAINCAADDQgABA0IAAUNCAAGDQgABw0IAAgNCAAJDQgACg0IAAsNCAAMDQgADQ0IAA4NCAAPDQgA
EA0IABENCAASDQgAEw0IABQNCAAVDQgAFg0IABcNCAAYDQgAGQ0IABoNCAAbDQgAHA0IAB0NCAAeDQgAHw0IACANCAAhDQgAIg0I
ACMNCAAkDQgAJQ0IACYNCAAnDQgAKA0IACkNCAAqDQgAKw0IACwNCAAtDQgALg0IAC8NCAAwDQgAMQ0IADINCAAzDQg8NDkBAAAA
AAA1OQEAAAAAABUAFYIDFYoDLBXqAhUQFQYVBhwYCDU5AQAAAAAAGAiBOAEAAAAAABYAKAg1OQEAAAAAABgIgTgBAAAAAAAREQAA
AMEB8MADAAAA6gIBCC8AAQIDBAUGBwgJCgsMDQ4PEBESExQVFhcYGRobHB0eHyAhIiMkJSYnKCkqKywtLi8wMTIzNDU2Nzg5Ojs8
PT4/QEFCQ0RFRkdISUpLTE1OT1BRUlNUVVZXWFlaW1xdXl9gYWJjZGVmZ2hpamtsbW5vcHFyc3R1dnd4eXp7fH1+f4CBgoOEhYaH
iImKi4yNjo+QkZKTlJWWl5iZmpucnZ6foKGio6SlpqeoqaqrrK2ur7CxsrO0AAAAFQQV0BEViglMFZoCFQASAADoCAgTFAAFAQTm
EwUHBADTDQgAwQ0IAPgNCADtDQgAHg0wAAQNCADPDRgAAQ0QAPcNEACQDQgABw0YANENEADSDQgAGg0YAJENEAAVDRAApg0QAMUN
CACTDQgAyg0IAPQNCADHDQgAjQ0IANsNCAASDUgAuQ0QAB0NEAD8DRAA+w0IAP4NCACwDQgA6A0IAJsNCADWDQgA4Q0IAPMNCAD2
DQgA5Q0IAOkNCAC8DQgACA1wAP0NEACMDQgA8Q0IAKINCACODQgAww0IALsNCACyDQgADg1IAKkNEADYDQgAtQ0IAN0NCAAbDSgA
lw0QAJ8NCADNDQgAGA0gAO4NEACsDQgApw0IAO8NCACtDQgADw0wAIkNEAD5DQgA7A0IALoNCAClDQgAjw0IAPANCADLDQgAvg0I
ALQNCACVDQgAxA0IAAwNaADjDRAA9Q0IAOINCADMDQgAwg0IAA0NMAAWDQgAvw0YAKgNCAAcDRgA/w0QAAANEADgDRAA8g0IAJwN
CACSDQgAyQ0IALENCADcDQgAEQ1AALcNEACdDQgAsw0IAMANCAC9DQgAuA0IABcNOAADDQgEKiNpWACrDSAAAg0YAIoNEAAKDRAA
1w0QABANEAC2DRAAxg0IAAkNGADqDRAACw0QAMgNEATeEwloABkNGACkDRAArg0IAPoNCAAUDSAAzg0QAJQNCADaDQgA5w0IANUN
CAAGDTAAlg0QAAUNEADfDRAAoQ0IAOsNCAApDfA8oBMAAAAAAACaEwAAAAAAABUAFYIDFYoDLBXqAhUQFQYVBhwYCCojAAAAAAAA
GAiJEwAAAAAAABYAKAgqIwAAAAAAABgIiRMAAAAAAAAREQAAAMEB8MADAAAA6gIBCC8AAQIDBAUGBwgDCQMKCwwNBQ4PEBESExQV
FhcYGRobHB0eHxEgISIjJCUmJygpKisaLC0uLzAxMjM0KjU2Nzg5OjUwOzw9Pj8zQEFCQ0RFRkdISQokSkRCS0xNTk9QUVJTVFVW
V1hZWlZbXF1eB19TYFJhYmNkZWZnaGlqEw9fax5sUG1ub3BxchdzC3RIdXZ3eEd5ejV7ZyZ8aH0qe35/T2CAKIGCP4NqhHCFhj+H
iImKOouMAAAAFQQVwBYV4hNMFegCFQASAACgC/A8AKwGEusrBgAAoWzo8y0GAABfm9OfMQYAAE1IvU0sBgAAlUpvETAGAABsmYaY
LwYAAPLjpgMuBgAAAzjeYwEoDFSx8tUBQAzu02xTATAMnp9zYwEwDI3fREkBEAxe6YcLATAMXvXRHAEYDHy+xOgBSAyeE2aIASAM
ddtwnwEQDCy4+vkBCAz1w7xyAQgMpkyZPAFgDOZlOmQBmAzrb2DCAUAMV9paIwE4DIYAMn8BeAwl/uJtAQgMjlyDewFoDHmnxCEB
QAyR+DlQAUAMWfCeAgFADLifGtQBKAw9jkuGASAM6vbpogEgDL55t8oBWAxz/vMNAVgMUodpegEwDKnW2WMBCAyCOKYPATAMSyWL
vwEICI23ugVYDCWq+EIBUAwfi9+nAUgMxyFt7wFADB08TecBKAyY9MVZAQgMtz441wFgDCLVeiABMAx+A+qNAVgMSmbK6AEwDF6J
U88BGAz+FPmxARAMvmVkJwEIDDxEtiwBKAxhPRczAUgMOeM4VgFIDFM7C3EBMAwU3hTSARgMg1nxhAH4DPgJnv8BOAwmUayKATgM
LprjOwEwDDZu4UgBMAw01dF4ARgMYuMYtgEYDGR3NUIBOAxRMsNnARgMNioJQgEYDIjM+E4BCAybow/mASAMOb0EwAVoCI8pGAEo
DKy1xKEB8AyNwGeAARAMqri4ggEoDFyid0kBaAwJLjSuAQgMnXX0KgEoDFwjdGsBCAw1s+iEARgMEzk+gAEIDBL3OEwBWAwnrPel
AUAMARxu6AEYLGKMkakqBgAA+579BgEYDFq2tqYBQAximRQTAQgMGp4yiQEoDAFyQzABEAx1NE4OAbAMTwRy+QEYDHMkrzcBEAwp
4ZG6AUgMzIljAAEoDJJbB2oBCAz5+wevAXgM7n5X9wFgDBNTr5gByAx34aJaAUAM9bxlpQEQDHvgn7UBQAwPPdDfAQgMbCWGTCFg
DLPCBm0BOAz+BwsDAVAM659suQEgDIopK4MBGAwkhWxOASgIFzl0JbgMjJtU+AEgCF+yLgWADM0DBVEBYAxQ1OiRAQgMwRYSPQEI
DAESD/QBQAyWcCY3AUAMpm3ajgFgDEFgRBEB0AzjsNJiAQgMLieVtAEwDIq8gQQBCAz1WPDpAbgM2avmBQFADCzS5H4BCAyG0nvq
ARgM4rwEKgFQDLqY5XMB+CzbsZ9ULQYAANkCYOsBGAxfuXcQAQgMibNDHgE4DN2YAm4BeAiG3ZplGAzvZ3+8AcAMMOcRcwGADK9X
ElgBKAi9LAxliAzuITkJAVgIp11ZZXgMpVF0NgF4DNah4hQBMAwzoEGHAQgIb3exBZAIPa5TRQAIXkrnRZAM+ZPK4AFgDHpcGN0B
mAxK2GarAWAMZhtk8gE4DIRZgjsBkAynIi5KARAMRYdS+AEoDETJQ+YBaAzEbAXuAYAMT1K3GwEQDAdwogwBQAyEvDcLARAMmxHk
lQEwDFARePIBCAxs7iKIAUgMlZ8LAQFYDD+EeUQBGAzHoH70AQgMNSUM2wEgDDCFea0hGAylnLExAWAMqrITegEgDKrKIAkBYAzy
uw7kAWAMo4Is0wEQDD+mbFsBCAxxj1lTAQgMrRshdgFIDAr1doMBOAzq3twTAXAMzXgwzgEQDD45ubkBCAwD0gQfATAMkJ4ZZgEQ
OA8xDWIsBgAAN0KZYCwGABUAFYIDFYoDLBXqAhUQFQYVBhwYCAAfi9+nMQYAGAgAYoyRqSoGABYAKAgAH4vfpzEGABgIAGKMkakq
BgAREQAAAMEB8MADAAAA6gIBCC8AAQIDBAUGBwgJCgsMDQ4PEBESExQVFhcYGRobHB0eHyAhIiMkJSYnKCkqKywtLi8wMTIzNDU2
Nzg5Ojs8PT4/QEFCQ0RFRkdISUpLTE1OT1BRUlNUVVZXWFlaW1xdXl9gYWJjZGVmZ2hpamtsbW5vcHFyc3R1dnd4eXp7fH1+f4CB
goOEhYaHiImKi4yNjo+QkZKTlJWWl5iZmpucnZ52n6ChoqOkpaanqKmqq6ytrq+wsbKzAAAAFQQVwAwVygxMFcACFQASAACgBvQf
AwAAHIhMAAAU4t8AAAaeIwAABLZ2AAACGRwAACQiMgAAFvrPAAAmFNQAACVyVAAAFKTIAAAPGQoAAAU9kgAAH3d3AAAjewIAABlk
lgAADx2IAAAGodoAAA+wSQAABXuEAAAHibAAABzCTgAAFNCNAAAgBN8AAAxfLwAAIXyoAAAMfzcAACCUCgAACP5YAAACgTQAAAHB
mwAABgYwAAATbjgAAAyDgwAAIi7IAAAe7+sAAABckwAAJQ5UAAAUVwcAABRt4gAAFNeMAAANHowAABGzowAAJiCLAAANT88AAB35
JQAAF1xGAAAUiNgAACHKMgAAFqG/AAAChhcAAB9chAAAAltSAAAgcRMAABiIeAAAALKVAAAh8aYAACWhygAADCwPAAAVDEcAACN0
KwAAB7gsAAACTHoAAACz+QAAAj6PAAAlmZYAAB38kAAAFyQ4AAATgPcAAADEygAAFkmZAAAOHg8AABWPiAAABUFPAAAMEKsAAAM3
WwAAC4mEAAAGr1QAACMqJgAAEcDsAAAOcj4AACPlcwAAEskRAAAC6FYAAAfYNAAAAMksAAAVTq8AABeEggAAAs2VAAAd0DoAACUH
4gAABnw6AAAd6EAAABjvNAAAEjbkAAAMqpQAACE+8AAAHixqAAAVhMoAABh2gAAAD68KAAAAwuwAABfWWAAAA0HwAAAE5jkAABFS
LgAAJXzfAAAd/ucAACCLcgAAE1LfAAALrqAAAA3jqgAAF/sQAAANfXYAABVBBAAAJWPfAAABQ2cAAASydgAACyIMAAAeqD4AAAI2
BAAACvnDAAAD8icAAByZfAAAHgeAAAAh8goAABxyOgAAFomIAAAKpWMAAA25FgAAFAggAAAWNe8AAB9IAgAAHcr0AAAaYCYAAAay
EgAAG3krAAAKB6AAABwsgAAABcU0AAAQhbAAAAeopQAAA/zvAAALGcwAAAzfUAAABtgmAAAfk/wAAAS04gAACbTjAAAJhYAAAAkV
yAAAHW2YAAASi6gAAABWrgAAB8D2AAAWH0gAABvYIAAABDW8AAARKK4AAB1VLgAAFOsUFQAVggMVigMsFeoCFRAVBhUGHBgFAAAm
IIsYBQAAAFauFgAoBQAAJiCLGAUAAABWrhERAAAAwQHwwAMAAADqAgEILwABAgMEBQYHCAkKCwwNDg8FEBESExQVFhcYGRobHB0e
HyAhEyIjJCUmJygpKissLS4vMDEyMzQ1Njc4OTo7PD0+PzNAQUJDREVGR0hJSktMTU5PDCZQUUhSU1RVVldYWVpbXF1eX2BhXWJj
ZGUHZmdoWWlqa2xtbm9wcXJzdHV2IHd4eXp7fH1+GX8NgE6BgoNkTYSFhodviImKi4yNjo9WaJAqkZKTlHKVfJaXmJmam5ydnp8A
AAAVBBVOFVJMFQgVABIAACeYBAAAAGNhcmQDAAAAdXBpCgAAAG5ldGJhbmtpbmcGAAAAd2FsbGV0FQAVbhVyLBXqAhUQFQYVBhw2
ACgGd2FsbGV0GARjYXJkEREAAAA32AMAAADqAgECL+SsYnsbeHPDYDsd+DGKg2ua0XtA+xfRXAYU6baleLabDV5aPgnl3poxfLse
mwMVBBVAFURMFQYVABIAACB8BwAAAFNVQ0NFU1MGAAAARkFJTEVEBwAAAFBFTkRJTkcVABVwFXQsFeoCFRAVBhUGHDYAKAdTVUND
RVNTGAZGQUlMRUQREQAAADjcAwAAAOoCAQIHAEQAEABAEgAFUQBAABIADwEAAQAEWBUAEAEUAABQJAAPGkUFAkAIAAQAARABAAAV
BBl8NQAYBnNjaGVtYRUMABUEJQIYCnBheW1lbnRfaWQAFQQlAhgIb3JkZXJfaWQAFQQlAhgHcGFpZF9hdCUUTIwSHCwAAAAAABUO
FQoVAhgGYW1vdW50JQoVBBUULFwVBBUUAAAAFQwlAhgGbWV0aG9kJQBMHAAAABUMJQIYBnN0YXR1cyUATBwAAAAW6gIZHBlsJgAc
FQQZNQAGEBkYCnBheW1lbnRfaWQVAhbqAhb4Ghb2DybwCyYIHBgINTkBAAAAAAAYCIE4AQAAAAAAFgAoCDU5AQAAAAAAGAiBOAEA
AAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgDqAgAAACYAHBUEGTUABhAZGAhvcmRlcl9pZBUCFuoCFvgVFroNJqoZJv4PHBgI
KiMAAAAAAAAYCIkTAAAAAAAAFgAoCCojAAAAAAAAGAiJEwAAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgDqAgAAACYAHBUE
GTUABhAZGAdwYWlkX2F0FQIW6gIW6BoWkhgmvDEmuB0cGAgAH4vfpzEGABgIAGKMkakqBgAWACgIAB+L36cxBgAYCABijJGpKgYA
EREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAOoCAAAAJgAcFQ4ZNQAGEBkYBmFtb3VudBUCFuoCFtAQFuIQJrZCJso1HBgFAAAmIIsY
BQAAAFauFgAoBQAAJiCLGAUAAABWrhERABksFQQVABUCABUAFRAVAgA8KQYZJgDqAgAAACYAHBUMGTUABhAZGAZtZXRob2QVAhbq
AhakAhasAiaaRyasRhw2ACgGd2FsbGV0GARjYXJkEREAGSwVBBUAFQIAFQAVEBUCADwW1BAZBhkmAOoCAAAAJgAcFQwZNQAGEBkY
BnN0YXR1cxUCFuoCFp4CFqYCJrhJJthIHDYAKAdTVUNDRVNTGAZGQUlMRUQREQAZLBUEFQAVAgAVABUQFQIAPBakExkGGSYA6gIA
AAAW6mAW6gImCBb2SgAZHBgMQVJST1c6c2NoZW1hGJgELy8vLy80Z0JBQUFRQUFBQUFBQUtBQXdBQmdBRkFBZ0FDZ0FBQUFBQkJB
QU1BQUFBQ0FBSUFBQUFCQUFJQUFBQUJBQUFBQVlBQUFBY0FRQUExQUFBQUp3QUFBQmdBQUFBTUFBQUFBUUFBQUFNLy8vL0FBQUJC
UkFBQUFBWUFBQUFCQUFBQUFBQUFBQUdBQUFBYzNSaGRIVnpBQURZLy8vL05QLy8vd0FBQVFVUUFBQUFIQUFBQUFRQUFBQUFBQUFB
QmdBQUFHMWxkR2h2WkFBQUJBQUVBQVFBQUFCZy8vLy9BQUFCQnhBQUFBQWdBQUFBQkFBQUFBQUFBQUFHQUFBQVlXMXZkVzUwQUFB
SUFBd0FCQUFJQUFnQUFBQUtBQUFBQWdBQUFKai8vLzhBQUFFS0VBQUFBQ0FBQUFBRUFBQUFBQUFBQUFjQUFBQndZV2xrWDJGMEFB
QUFCZ0FJQUFZQUJnQUFBQUFBQWdETS8vLy9BQUFCQWhBQUFBQWNBQUFBQkFBQUFBQUFBQUFJQUFBQWIzSmtaWEpmYVdRQUFBQUF2
UC8vL3dBQUFBRkFBQUFBRUFBVUFBZ0FCZ0FIQUF3QUFBQVFBQkFBQUFBQUFBRUNFQUFBQUNRQUFBQUVBQUFBQUFBQUFBb0FBQUJ3
WVhsdFpXNTBYMmxrQUFBSUFBd0FDQUFIQUFnQUFBQUFBQUFCUUFBQUFBPT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9uIDI1
LjAuMRlsHAAAHAAAHAAAHAAAHAAAHAAAAGAFAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "001-orders-payments-join"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

orders = spark.read.parquet((_data_dir / "orders.parquet").resolve().as_posix())
orders.createOrReplaceTempView("orders")
payments = spark.read.parquet((_data_dir / "payments.parquet").resolve().as_posix())
payments.createOrReplaceTempView("payments")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        return passed

    return check

EXPECTED = {'column_fingerprints': {'customer_id': '002d7e3b5cffbb1f1a37ab674e245518240b0ce849ff33c3f92f36c78119e86b',
                         'paid_orders': 'b353906e9034cd6d3187362835cb95b587d33f2a614d23ca3ec231a0f9b0796f',
                         'revenue': '0ead75a7dc8ef562852a3c7276eb7e78a82249662f5c35f9a64cbea4af0f4c12'},
 'columns': [['customer_id', 'bigint'], ['paid_orders', 'bigint'], ['revenue', 'decimal(12,2)']],
 'fingerprint': 'fd9894d4b3812019c3906fee9e19f66aa92e1f4ea1cf133283b2e1d3423c3e47',
 'order_matters': False,
 'row_count': 39}

check = make_check(EXPECTED)

## The situation

A teammate asked an AI assistant for a "revenue per customer" report and shipped what
it wrote. The job runs without errors and the output looks reasonable. Finance has now
compared it to the ledger: the report's total revenue is too high, and only some
customers are off.

The AI-written code is in the answer cell. Find the bug and fix it.

## Input tables

### `orders`

One row per order.

| Column | Type | Description |
|---|---|---|
| `order_id` | BIGINT | Unique per order. |
| `customer_id` | BIGINT | The customer who placed the order. |
| `order_date` | DATE | Date the order was placed. |
| `order_amount` | DECIMAL(10,2) | Full value of the order in rupees. |

### `payments`

One row per payment attempt. An order can have no rows here, one row, or several:
a failed attempt followed by a retry, or an order paid in two or three parts.

| Column | Type | Description |
|---|---|---|
| `payment_id` | BIGINT | Unique per payment attempt. |
| `order_id` | BIGINT | The order the attempt was for. A few rows point to orders that are not in `orders`. |
| `paid_at` | TIMESTAMP_NTZ | When the attempt happened. |
| `amount` | DECIMAL(10,2) | Amount of this attempt in rupees. |
| `method` | STRING | `upi`, `card`, `netbanking` or `wallet`. |
| `status` | STRING | `SUCCESS`, `FAILED` or `PENDING`. |

## Requirement

- An order is **paid** if it has at least one `SUCCESS` payment.
- A customer's **revenue** is the sum of `order_amount` over their paid orders.
  Each paid order counts once, however many payments it took.
- Every customer in `orders` must appear in the result. A customer with no paid
  orders has `paid_orders` = 0 and `revenue` = 0.00.

## Expected output

One row per customer.

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | |
| `paid_orders` | BIGINT | Number of paid orders. |
| `revenue` | DECIMAL(12,2) | Sum of `order_amount` over paid orders. |

In [ ]:
orders.show(5)
payments.show(5)

## Your answer

The cell below is the AI-written code. It runs, but its output is wrong. Find the bug and fix it. The tables are available as DataFrames and as views: `orders`, `payments`.

In [ ]:
from pyspark.sql import functions as F


def solve(spark):
    """Revenue per customer from paid orders."""
    orders = spark.table("orders")
    payments = spark.table("payments")

    # Only successful payments count towards a paid order
    successful_payments = payments.filter(F.col("status") == "SUCCESS")

    # Left join so that customers without any paid orders are still included
    orders_with_payments = orders.join(successful_payments, on="order_id", how="left")

    # Aggregate per customer
    result = orders_with_payments.groupBy("customer_id").agg(
        F.countDistinct(
            F.when(F.col("payment_id").isNotNull(), F.col("order_id"))
        ).alias("paid_orders"),
        F.coalesce(
            F.sum(F.when(F.col("payment_id").isNotNull(), F.col("order_amount"))),
            F.lit(0),
        )
        .cast("decimal(12,2)")
        .alias("revenue"),
    )

    return result


result = solve(spark)
result.show()

In [ ]:
check(result)